# 第 17 週 Capstone｜Capstone:最佳化即動力學

整學期在這裡合體。你會親手證明:梯度下降<strong>就是</strong>用一階數值方法解一個微分方程,而 learning rate 就是步長。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Part 1｜梯度下降就是 Euler 法(逐位元驗證)

W16 觀念 6 說梯度下降<strong>就是</strong>把 Euler 法套在梯度流上。這不是類比——這一格用兩支獨立寫出的程式,證明它們產生逐位元相同的軌跡。


In [ ]:
# ============ 本 capstone 共用的兩個解算器(W14/W15 寫過,這裡重用) ============

def euler(f, y0, t0, t1, h):
    """顯式尤拉法解 y' = f(t, y)"""
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h*f(t, y)); ts.append(t + h)
    return np.array(ts), np.array(ys)

def rk4(f, y0, t0, t1, h):
    """四階 Runge-Kutta"""
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        k1 = f(t,       y)
        k2 = f(t + h/2, y + h*k1/2)
        k3 = f(t + h/2, y + h*k2/2)
        k4 = f(t + h,   y + h*k3)
        ys.append(y + h*(k1 + 2*k2 + 2*k3 + k4)/6); ts.append(t + h)
    return np.array(ts), np.array(ys)

def gradient_descent(dL, theta0, eta, steps):
    """教科書版梯度下降 —— 刻意不參考上面兩支的寫法"""
    th = [theta0]
    for _ in range(steps):
        th.append(th[-1] - eta*dL(th[-1]))
    return np.array(th)

# ============ 損失函數:L(θ) = A/2 · θ² ============
A  = 4.0
L  = lambda th: A/2 * th**2
dL = lambda th: A * th
theta_exact = lambda t, th0: th0*np.exp(-A*t)      # 梯度流的解析解

# ---- 逐位元比較 ----
eta, steps = 0.15, 15
_, path_euler = euler(lambda t, th: -dL(th), 1.0, 0.0, eta*steps, eta)
path_gd = gradient_descent(dL, 1.0, eta, steps)

print(f"L(θ) = {A}/2·θ²   梯度流 θ' = -{A}θ   η = h = {eta}\n")
print(f"{'n':>3} {'Euler 解梯度流':>20} {'梯度下降':>20} {'相同?':>7}")
for n in range(0, len(path_gd), 3):
    print(f"{n:3d} {path_euler[n]:20.16f} {path_gd[n]:20.16f} "
          f"{str(path_euler[n] == path_gd[n]):>7}")
print(f"\n★ 全部 {len(path_gd)} 個點逐位元相同? "
      f"{np.array_equal(path_euler, path_gd)}")
print("  → 「梯度下降就是 Euler 法」不是類比,是同一個演算法")

# ---- 離散步伐 vs 連續軌跡 ----
tt = np.linspace(0, eta*steps, 300)
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(tt, theta_exact(tt, 1.0), 'k-', lw=2, label='gradient flow (exact)')
for e, c in [(0.05, 'C0'), (0.15, 'C1'), (0.45, 'C3')]:
    p = gradient_descent(dL, 1.0, e, int(2.25/e))
    plt.plot(np.arange(len(p))*e, p, 'o--', ms=4, color=c, label=f'GD η={e}')
plt.xlabel('t = n·η'); plt.ylabel('θ'); plt.legend(fontsize=8)
plt.title('Larger η = coarser Euler steps')

# ---- 損失單調下降(W16 觀念 5 的 dL/dt = -(L')² ≤ 0)----
plt.subplot(1, 2, 2)
for e, c in [(0.05, 'C0'), (0.15, 'C1'), (0.45, 'C3')]:
    p = gradient_descent(dL, 1.0, e, int(2.25/e))
    plt.semilogy(np.arange(len(p))*e, L(p), 'o-', ms=3, color=c, label=f'η={e}')
plt.xlabel('t = n·η'); plt.ylabel('L(θ)'); plt.legend(fontsize=8)
plt.title('Loss decreases monotonically (full-batch)')
plt.tight_layout(); plt.show()

In [ ]:
# TODO 學生練習:把 A 改成 20,再把 η 設成 0.11(> 2/A = 0.1)
# 兩支程式還是逐位元相同嗎?軌跡會發生什麼事?
# (提示:相同性與穩定性是兩回事 —— 它們會「一起」發散)

## Part 2｜momentum 是帶阻尼的二階 ODE(而且不一定更快)

W16 觀念 8 說 momentum 對應 $\theta''+\gamma\theta'+L'=0$,等效阻尼 $\gamma\approx\frac{1-\beta}{\eta}$,臨界值是 $2\sqrt A$。這一格驗證這個對應——並發現一件反直覺的事。


In [ ]:
def momentum_gd(dL, theta0, eta, beta, steps):
    """標準 momentum:v ← βv - η∇L,  θ ← θ + v"""
    th, v, path = theta0, 0.0, [theta0]
    for _ in range(steps):
        v = beta*v - eta*dL(th)
        th = th + v
        path.append(th)
    return np.array(path)

A, eta = 4.0, 0.2
dL = lambda th: A*th
gamma_crit = 2*math.sqrt(A)

print(f"A = {A},  η = {eta},  臨界阻尼 2√A = {gamma_crit}")
print(f"等效阻尼 γ ≈ (1-β)/η\n")
print(f"{'β':>6} {'γ≈(1-β)/η':>12} {'vs 臨界':>10} {'到 |θ|<1e-6 的步數':>20}")

results = {}
for beta in [0.0, 0.3, 0.5, 0.9, 0.99]:
    g = (1-beta)/eta
    path = momentum_gd(dL, 1.0, eta, beta, 3000)
    hit = np.argmax(np.abs(path) < 1e-6) if np.any(np.abs(path) < 1e-6) else None
    results[beta] = path
    kind = "過阻尼" if g > gamma_crit else "欠阻尼"
    print(f"{beta:6.2f} {g:12.2f} {kind:>10} {hit if hit else '未達成':>20}")

print("\n★ 反直覺的發現:在這個「單變數、條件良好」的問題上,")
print("  β 越大反而越慢 —— 因為 γ=(1-β)/η 遠低於臨界阻尼,嚴重欠阻尼、一直震盪。")
print("  momentum 不是「免費加速」,它是「加入慣性」,而慣性會超調。\n")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for beta, c in [(0.0, 'C0'), (0.5, 'C1'), (0.9, 'C3')]:
    ax[0].plot(results[beta][:60], 'o-', ms=3, color=c, label=f'β={beta}')
ax[0].axhline(0, color='k', lw=0.6); ax[0].legend(fontsize=8)
ax[0].set_xlabel('step'); ax[0].set_ylabel('θ')
ax[0].set_title('1-D well-conditioned: momentum OVERSHOOTS')

# ---- 那 momentum 什麼時候才有用?條件數大的時候 ----
# L(x,y) = (x² + κ·y²)/2,κ 是條件數
kappa = 50.0
dL2 = lambda p: np.array([p[0], kappa*p[1]])
eta2 = 0.9 * 2/kappa                      # η 被最陡方向綁住

def momentum_2d(beta, steps=400):
    p, v, path = np.array([10.0, 1.0]), np.zeros(2), []
    for _ in range(steps):
        v = beta*v - eta2*dL2(p)
        p = p + v
        path.append(p.copy())
    return np.array(path)

print(f"改成 2 維 ill-conditioned:L = (x² + {kappa}y²)/2,條件數 κ = {kappa}")
print(f"η 被最陡方向綁住:η = 0.9·2/κ = {eta2:.4f}")
print(f"{'β':>6} {'400 步後 |x|(平坦方向)':>26}")
for beta, c in [(0.0, 'C0'), (0.9, 'C3')]:
    path = momentum_2d(beta)
    ax[1].semilogy(np.abs(path[:, 0]), color=c,
                   label=f"{'plain GD' if beta == 0 else f'momentum β={beta}'}")
    print(f"{beta:6.2f} {abs(path[-1, 0]):26.4e}")
ax[1].set_xlabel('step'); ax[1].set_ylabel('|x|  (flat direction)')
ax[1].legend(fontsize=8); ax[1].set_title(f'2-D ill-conditioned (κ={kappa}): momentum WINS')
plt.tight_layout(); plt.show()

print("\n★ 結論:momentum 的價值不在「一律更快」,而在<條件數大>的時候。")
print("  平坦方向靠慣性累積速度,陡峭方向的來回梯度互相抵消。")

In [ ]:
# TODO 學生練習:在 1 維那個問題上,用理論找出「最好的 β」
# 提示:臨界阻尼 γ = 2√A,而 γ ≈ (1-β)/η → 解出 β
# 用你算出的 β 跑一次,步數比 β=0 少嗎?

## Part 3｜用 RK4 解梯度流:為什麼 ML 不這樣做

既然梯度下降是「一階」的 Euler,為什麼不用四階的 RK4?這一格量出精度差距,然後回答那個更重要的問題:<strong>為什麼實務上不用</strong>。


In [ ]:
A = 4.0
dL = lambda th: A*th
f  = lambda t, th: -dL(th)                 # 梯度流
exact = lambda t: math.exp(-A*t)           # θ0 = 1
T = 2.0

print(f"追蹤梯度流 θ' = -{A}θ 到 t = {T}(精確值 {exact(T):.10e})\n")
print(f"{'h':>8} {'Euler(=GD)':>14} {'誤差':>11} {'比值':>7} "
      f"{'RK4':>14} {'誤差':>11} {'比值':>7}")
pe = pr = None
hs, ee, er = [], [], []
for h in [0.2, 0.1, 0.05, 0.025, 0.0125]:
    ve = euler(f, 1.0, 0, T, h)[1][-1]
    vr = rk4(f, 1.0, 0, T, h)[1][-1]
    e1, e2 = abs(ve - exact(T)), abs(vr - exact(T))
    hs.append(h); ee.append(e1); er.append(e2)
    r1 = f"{pe/e1:7.2f}" if pe else "      -"
    r2 = f"{pr/e2:7.2f}" if pr else "      -"
    print(f"{h:8.4f} {ve:14.10f} {e1:11.3e} {r1} {vr:14.10f} {e2:11.3e} {r2}")
    pe, pr = e1, e2

plt.loglog(hs, ee, 'o-', label='Euler (= gradient descent)')
plt.loglog(hs, er, 's-', label='RK4')
plt.loglog(hs, np.array(hs)*ee[0]/hs[0], 'k:', lw=1, label='slope 1')
plt.loglog(hs, np.array(hs)**4*er[0]/hs[0]**4, 'k--', lw=1, label='slope 4')
plt.xlabel('h  (= η)'); plt.ylabel(f'|error at t={T}|'); plt.legend(fontsize=8)
plt.title('Tracking the gradient flow: Euler vs RK4')
plt.show()

for name, e in [('Euler', ee), ('RK4', er)]:
    print(f"{name:6s} log-log 斜率 = {np.polyfit(np.log10(hs), np.log10(e), 1)[0]:.3f}")

# ---- 關鍵問題:成本 ----
print("\n" + "="*62)
print("RK4 精確得多。那為什麼訓練神經網路不用它?\n")
print("① 每步要算 4 次梯度。反向傳播是訓練最貴的操作 —— 成本直接 ×4。")
print("② ML 的目標不是「精確追蹤軌跡」,只是「到達低點」。")
print("   走哪條路無所謂,精度花在軌跡上是浪費。")
print("③ 真實梯度有 mini-batch 雜訊(W12 觀念 8)。")
print("   用四階方法精確積分一個帶雜訊的場,像用游標卡尺量海浪。")

# 用「等成本」來比:給定固定的梯度計算次數,誰走得遠?
budget = 240                                # 允許 240 次梯度計算
print(f"\n等成本比較:預算 {budget} 次梯度計算")
h_e = T/budget                              # Euler 每步 1 次
h_r = T/(budget//4)                         # RK4 每步 4 次
ve = euler(f, 1.0, 0, T, h_e)[1][-1]
vr = rk4(f, 1.0, 0, T, h_r)[1][-1]
print(f"  Euler h={h_e:.5f}({budget} 步)   誤差 = {abs(ve-exact(T)):.3e}")
print(f"  RK4   h={h_r:.5f}({budget//4} 步) 誤差 = {abs(vr-exact(T)):.3e}")
print("\n→ 即使算等成本,RK4 在「追蹤軌跡」這件事上仍然大勝。")
print("  所以不用它的理由是 ② 和 ③ ——「不需要那個精度」,不是「負擔不起」。")

In [ ]:
# TODO 學生練習:把 exact 換成 L(θ) 的值,比較「損失下降得多快」而不是「軌跡多準」
# RK4 的優勢還那麼明顯嗎?這說明了什麼?

## Part 4｜用泰勒收攏一切:learning rate 的完整圖像

最後一格把整學期串起來:用<strong>泰勒展開</strong>解釋誤差階數與穩定門檻,並把 learning rate 的每一個現象對應到一個數學事實。


In [ ]:
A = 4.0
dL = lambda th: A*th

# ============ ① 泰勒預測單步誤差 ============
# θ(t+h) = θ(t) + hθ'(t) + (h²/2)θ''(ξ);Euler 只取前兩項
# 對 θ' = -Aθ 而言 θ'' = A²θ,故單步誤差 ≈ (h²/2)A²θ
print("① 泰勒預測 Euler 的單步誤差 ≈ (h²/2)·A²·θ\n")
print(f"{'h':>8} {'實測單步誤差':>16} {'泰勒預測':>14} {'比值':>8}")
for h in [0.1, 0.05, 0.025, 0.0125]:
    one_step  = 1.0 - h*A*1.0                  # Euler 走一步
    true_step = math.exp(-A*h)                 # 精確走一步
    measured  = abs(one_step - true_step)
    predicted = h**2/2 * A**2 * 1.0
    print(f"{h:8.4f} {measured:16.3e} {predicted:14.3e} {measured/predicted:8.4f}")
print("  → 比值趨近 1,泰勒的二階項確實是單步誤差的主項\n")

# ============ ② 穩定門檻 η < 2/A ============
print(f"② 穩定門檻:θ_{{n+1}} = (1-ηA)θ_n,需 |1-ηA| < 1 → η < 2/A = {2/A}\n")
print(f"{'η':>7} {'公比 1-ηA':>12} {'|公比|':>9} {'行為':>18}")
for e in [0.1, 0.25, 0.3, 0.49, 0.5, 0.51]:
    q = 1 - e*A
    if abs(q) >= 1:
        beh = "發散"
    elif q > 0:
        beh = "單調收斂"
    elif q == 0:
        beh = "一步到位(最佳)"
    else:
        beh = "震盪收斂"
    print(f"{e:7.3f} {q:12.3f} {abs(q):9.3f} {beh:>18}")
print(f"\n  最佳 η = 1/A = {1/A}(公比 0,一步到位)")
print(f"  這正是牛頓法在二次函數上的行為:η = 1/L''\n")

# ============ ③ 整學期的對應表 ============
print("③ learning rate 的每一個現象,都對應一個數學事實\n")
rows = [
    ("η 就是步長 h",            "梯度下降 = Euler 法解梯度流",        "W14 + W16"),
    ("η 太大會發散",            "違反 |1-ηA| < 1 的穩定條件",         "W15 + W16"),
    ("η 太小收斂慢",            "公比接近 1,每步只縮一點",            "W16"),
    ("最佳 η ≈ 1/L''",         "公比為 0;等於牛頓法",                "W4 + W16"),
    ("η 上限由曲率決定",         "2/λ_max,λ 是 Hessian 特徵值",       "→ 線性代數"),
    ("learning rate decay",     "自適應步長:難處用小步",              "W15"),
    ("Adam 各方向不同步長",      "預條件,壓低條件數",                  "→ 線性代數"),
    ("momentum 會超調",         "二階 ODE 欠阻尼",                    "W16"),
    ("loss 單調下降(全批次)",   "dL/dt = -(L')² ≤ 0(鏈鎖法則)",      "W2 + W16"),
    ("mini-batch 讓 loss 抖",   "期望損失的蒙地卡羅估計有變異數",       "W12"),
]
print(f"{'ML 的現象':<26} {'數學事實':<36} {'哪一週'}")
print("-"*82)
for a, b, c in rows:
    print(f"{a:<26} {b:<36} {c}")

print("\n" + "="*82)
print("你在銜接課手刻的梯度下降,數學上是「用一階數值方法解一個微分方程」。")
print("整個學期的極限、導數、泰勒、積分、微分方程,在這一頁合體了。")